# RSNA Knee: the real score of every public notebook, and why the leaderboard is a tie-break lottery

There are well over a thousand public notebooks in this competition. Many put a score in their title,
and some of those titles are not what the notebook actually scored. Before spending a 9-hour GPU run on
a fork, it is worth knowing the **real** number.

This notebook pulls three things live from Kaggle, so a fork always shows today's numbers:

1. **The shape of the public leaderboard.** Scores are shown to three decimals, so hundreds of teams
   share the exact same score. Inside such a block, the team that submitted first ranks first.
2. **Titles against real scores.** Each public notebook's best public score, set against the score its
   title or URL claims.
3. **The best public notebooks by real score**, with votes and date.

Nothing here touches competition data or labels: it only reads the same public pages you can see in
the browser. Turn **Internet on** in the notebook settings to run it.

To use it for another competition, change `COMPETITION` in the next cell.

In [ ]:
COMPETITION = "rsna-knee-abnormality-detection"

import re
import time

import matplotlib.pyplot as plt
import pandas as pd
import requests

pd.set_option("display.max_colwidth", 70)

# Kaggle's web pages load their data from these endpoints. A first GET sets the anti-forgery
# cookie that the POSTs must echo back. No login or API key is needed for public data.
_session = requests.Session()
_session.get("https://www.kaggle.com/competitions", timeout=60)
_headers = {"x-xsrf-token": _session.cookies.get("XSRF-TOKEN", "")}


def kaggle_api(service, method, body, tries=4):
    for attempt in range(tries):
        r = _session.post(f"https://www.kaggle.com/api/i/{service}/{method}", json=body,
                          headers=_headers, timeout=90)
        if r.status_code == 200:
            return r.json()
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"{method} failed: HTTP {r.status_code}. Is Internet turned on?")


comp = kaggle_api("competitions.CompetitionService", "GetCompetition", {"competitionName": COMPETITION})
comp = comp.get("competition", comp)
higher_is_better = comp.get("evaluationAlgorithm", {}).get("isMax", True)
print(f"{comp['title']}: {comp['totalTeams']:,} teams, deadline {comp['deadline'][:10]}")

## 1. The leaderboard is a handful of giant tie blocks

Kaggle's medal cut-offs come from the team count: for 1,000 or more teams, bronze is the top 10% and
silver the top 5%. The chart shows how many teams sit at each displayed score near the top, and where
the cut-offs fall today.

In [ ]:
lb = kaggle_api("competitions.LeaderboardService", "GetLeaderboard", {"competitionId": comp["id"]})
board = pd.DataFrame(lb["publicLeaderboard"])
board["score"] = board["displayScore"].astype(float)
n = int(comp["totalTeams"])
cuts = {"gold": 10 + n // 500, "silver": n // 20, "bronze": n // 10} if n >= 1000 else \
       {"gold": 10, "silver": 50, "bronze": 100}

blocks = (board.groupby("score")
          .agg(teams=("rank", "size"), best_rank=("rank", "min"), worst_rank=("rank", "max"))
          .sort_index(ascending=not higher_is_better).reset_index())
blocks["medal cut inside"] = [
    " + ".join(m for m, r in cuts.items() if lo <= r <= hi)
    for lo, hi in zip(blocks.best_rank, blocks.worst_rank)]
top = blocks.head(18)

INK, INK2, MUTED, GRID, BLUE, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#2a78d6", "#fcfcfb"
fig, ax = plt.subplots(figsize=(11, 6.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.barh([f"{s:.3f}" for s in top["score"]], top["teams"], color=BLUE, height=0.72)
ax.invert_yaxis()
for y, (teams, lo, hi, cut) in enumerate(zip(top.teams, top.best_rank, top.worst_rank,
                                             top["medal cut inside"])):
    label = f"{teams:,} teams, ranks {lo:,}-{hi:,}" + (f"   <- {cut} cut-off inside" if cut else "")
    ax.text(teams + top.teams.max() * 0.01, y, label, va="center", fontsize=9.5, color=INK2)
ax.set_xlabel("teams with exactly this displayed score", color=INK2)
ax.set_ylabel("public score", color=INK2)
ax.set_title(f"How many teams share each score (top {len(top)} scores, {n:,} teams)", loc="left",
             fontsize=13, color=INK, fontweight="bold")
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(MUTED)
ax.tick_params(colors=INK2)
ax.set_xlim(0, top.teams.max() * 1.55)
plt.tight_layout()
plt.show()

print("Medal cut-offs today:", {m: f"rank {r:,}, score {board.loc[board['rank'] == r, 'displayScore'].iloc[0]}"
                                for m, r in cuts.items() if (board["rank"] == r).any()})
top

### What the tie blocks mean for you

- **A copied public notebook lands inside its score's block, behind everyone who submitted it
  earlier.** Ties are broken by submission time. That is why two teams with the same score can be hundreds
  of ranks apart, and why the block's size matters more than its score.
- **A small real gain jumps a whole block.** Moving up one displayed step (0.001 here) skips every
  team in the block you leave.
- **The private leaderboard is scored on different studies.** Identical notebooks stay tied there too,
  but the blocks will sit at different scores, so medal lines move.

## 2. Titles against real scores

Every public notebook that has been submitted carries its best public score. Where a title or URL
contains a score (for example `...-0-957-...`), we compare the two.

In [ ]:
def list_notebooks(sort_by):
    out, page = [], 1
    while page < 40:
        body = {"kernelFilterCriteria": {"search": "", "listRequest": {
                    "competitionId": comp["id"], "sortBy": sort_by, "pageSize": 100,
                    "group": "EVERYONE", "page": page}},
                "detailFilterCriteria": {"deletedAccessBehavior": "RETURN_NOTHING",
                                         "unauthorizedAccessBehavior": "RETURN_NOTHING",
                                         "excludeResultsFilesOutputs": True},
                "readMask": "pinnedKernels"}
        got = kaggle_api("kernels.KernelsService", "ListKernels", body).get("kernels", [])
        if not got:
            break
        out += got
        page += 1
    return out


raw = {}
for sort_by in ("SCORE_DESCENDING", "DATE_RUN", "VOTE_COUNT"):
    for k in list_notebooks(sort_by):
        raw[k["scriptUrl"]] = k
nbs = pd.DataFrame([{
    "notebook": k["scriptUrl"].removeprefix("/code/"),
    "title": k.get("title", ""),
    "real score": k.get("bestPublicScore"),
    "votes": k.get("totalVotes") or 0,
    "updated": str(k.get("scriptVersionDateCreated", ""))[:10],
} for k in raw.values()])


def claimed(row):
    found = re.findall(r"0\.9\d{1,3}", row["title"] + " " + row["notebook"].replace("-", "."))
    return max(map(float, found)) if found else None


nbs["claimed"] = nbs.apply(claimed, axis=1)
scored = nbs.dropna(subset=["real score"])
claims = scored.dropna(subset=["claimed"]).copy()
claims["overstated by"] = (claims["claimed"] - claims["real score"]).round(3)
over = claims[claims["overstated by"] >= 0.002].sort_values("overstated by", ascending=False)
print(f"{len(nbs):,} public notebooks, {len(scored):,} with a leaderboard score, "
      f"{len(claims)} put a score in the title or URL.")
print(f"{len(over)} of those {len(claims)} ({len(over) / max(len(claims), 1):.0%}) claim at least 0.002 "
      f"more than they ever scored.")
over[["notebook", "claimed", "real score", "overstated by", "votes", "updated"]].head(15)

## 3. The best public notebooks by real score

Sorted by real best public score, then votes. A high score with few votes is often brand new and not
yet copied by many teams.

In [ ]:
best = (scored.sort_values(["real score", "votes"], ascending=[not higher_is_better, False])
        .head(20)[["notebook", "real score", "votes", "updated", "title"]].reset_index(drop=True))
best.index += 1
best

## Before you fork: a short checklist

1. **Use the real score, not the title.** Section 2 shows how often they differ.
2. **Open the notebook's Input panel.** If an input is private, your copy runs without it and usually
   scores lower than the author's. Several high-scoring notebooks here blend in the author's own private
   model at the end.
3. **Check the runtime.** The limit is 9 hours. A heavy stack plus an extra model can time out on the
   hidden test even when the quick save run finishes.
4. **Submit early when your score sits in a big block.** Inside a block, earlier submissions rank first.
5. **Do not chase 0.001 moves on the public board alone.** The public board is 30% of the test set;
   the private 70% decides medals.

If this saved you a wasted GPU run, an upvote helps others find it. Corrections are welcome in the
comments.